# Inspect your WLASL landmark data
Run setup, manifest and extraction first. Select the project's Python environment as the notebook kernel. These cells inspect real cached data; no results are precomputed.
Install the optional notebook tools with `python -m pip install jupyterlab ipykernel`, then run `python -m jupyterlab`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from signlang.data import read_manifest
from signlang.features import prepare
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
rows = read_manifest(ROOT / 'data/processed/manifest.csv')
coverage = json.loads((ROOT / 'data/processed/coverage.json').read_text())
coverage['counts']

## Inspect one available sample
Change the selected row to inspect different classes and splits. Check the matching original video to confirm the sign is complete.

In [ ]:
row = next(r for r in rows if (ROOT / 'data/landmarks' / (r['video_id'] + '.npz')).exists())
with np.load(ROOT / 'data/landmarks' / (row['video_id'] + '.npz')) as data:
    points, valid = data['points'], data['valid']
print(row['gloss'], row['split'], points.shape, valid.shape)
features = prepare(points, valid)
print('GRU input:', features.shape)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(valid[:, 33:54].mean(axis=1), label='Left hand')
ax.plot(valid[:, 54:75].mean(axis=1), label='Right hand')
ax.set(xlabel='Frame', ylabel='Fraction of landmarks observed', title=row['gloss'])
ax.legend();

In [ ]:
frame = len(points) // 2
fig, ax = plt.subplots(figsize=(5, 5))
for start, stop, color, name in [(0,33,'gray','Pose'),(33,54,'blue','Left hand'),(54,75,'orange','Right hand')]:
    p = points[frame, start:stop][valid[frame, start:stop]]
    ax.scatter(p[:,0], p[:,1], c=color, label=name)
ax.invert_yaxis()
ax.set(xlabel='Normalized image x', ylabel='Normalized image y', title='Observed keypoints')
ax.legend();

## Questions to answer before training
- Are both shoulders visible through most of the clip?
- Are hand slots stable when hands cross?
- Is the original video already trimmed correctly?
- Which classes have missing training or test examples?
- Are signs recognizable to a person familiar with ASL?
Write observations in your report; do not treat a successful extraction as proof that the label is correct.